# Sentiment Analysis on Asha Bhosle Tweets

This notebook demonstrates data loading, preprocessing, model training, and evaluation.

In [ ]:
import re, json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, classification_report, confusion_matrix

In [ ]:
BASE_DIR = Path(r'/mnt/data/asha_bhosle_sentiment_repo')
DATA_PATH = BASE_DIR / 'data' / 'asha_bhosle_tweets.csv'
RESULTS_DIR = BASE_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#', ' ', text)
    text = re.sub(r'[^a-z\s\']', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df = pd.read_csv(DATA_PATH)
df['clean_tweet'] = df['tweet'].apply(clean_text)
df.head()

In [ ]:
df['sentiment'].value_counts().plot(kind='bar'); plt.title('Label Distribution'); plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df['clean_tweet'], df['sentiment'], test_size=0.2, random_state=42, stratify=df['sentiment'])
models = {
    'Naive Bayes': MultinomialNB(),
    'SVM': LinearSVC(),
    'Logistic Regression': LogisticRegression(max_iter=2000)
}
results = []
for name, clf in models.items():
    pipe = Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2), stop_words='english')), ('clf', clf)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    p, r, f1, _ = precision_recall_fscore_support(y_test, pred, average='macro', zero_division=0)
    results.append({'Model': name, 'Precision': p, 'Recall': r, 'F1': f1})
    print(name)
    print(classification_report(y_test, pred, zero_division=0))
results_df = pd.DataFrame(results)
results_df

In [ ]:
results_df[['Model', 'Precision', 'Recall']].plot(x='Model', kind='bar', title='Precision vs Recall'); plt.ylim(0, 1.05); plt.show()